# Process Mining Group 5 notebook

## Setup

Shared imports, constants, sub-process scoping (`SUBPROCESSES`), miner settings, and helper
functions used by every section below. Run this cell first.


In [70]:
import os
import re
import pandas as pd
import matplotlib
import pm4py
from pm4py.algo.filtering.dfg import dfg_filtering
from pm4py.visualization.dfg import visualizer as dfg_visualizer
from pm4py.visualization.dfg.variants.frequency import Parameters as DFGParam
from pm4py.visualization.petri_net import visualizer as pn_visualizer
from pm4py.visualization.petri_net.common.visualize import Parameters as PNParam

CASE_ID = "case:concept:name"
ACTIVITY = "concept:name"
TIMESTAMP = "time:timestamp"
LIFECYCLE = "lifecycle:transition"
MIN_ACTIVITY_FREQ = 0.01
COLLAPSE_SELF_LOOPS = True  # collapse consecutive repeats of an activity (A,A->A) to remove self-loop redo constructs

XES_PATH = "PermitLog.xes"
OUTPUT_FOLDER = "Output"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

SUBPROCESSES = {
    "permit":              {"prefixes": ["Permit"],              "extra": ["Start trip", "End trip"],             "tail": ["Start trip", "End trip"],             "start": ["Permit SUBMITTED by EMPLOYEE"],              "end": ["FINAL_APPROVED", "REJECTED", "End trip"]},
    "declaration":         {"prefixes": ["Declaration"],         "extra": ["Send Reminder"],                      "tail": [],                                     "start": ["Declaration SUBMITTED by EMPLOYEE"],         "end": ["FINAL_APPROVED", "REJECTED"]},
    "request_for_payment": {"prefixes": ["Request For Payment"], "extra": ["Request Payment", "Payment Handled"],  "tail": ["Request Payment", "Payment Handled"],  "start": ["Request For Payment SUBMITTED by EMPLOYEE"], "end": ["FINAL_APPROVED", "REJECTED", "Payment Handled"]},
}

BASE_SAMPLE_SIZES = [250, 500, 1000]

heuristics_settings = [
    {"dependency_threshold": 0.20, "and_threshold": 0.40, "loop_two_threshold": 0.20},
    {"dependency_threshold": 0.30, "and_threshold": 0.50, "loop_two_threshold": 0.30},
    {"dependency_threshold": 0.40, "and_threshold": 0.60, "loop_two_threshold": 0.40},
]

inductive_settings = [
    {"noise_threshold": 0.00},
    {"noise_threshold": 0.05},
    {"noise_threshold": 0.10},
]

DECISION_METRICS = ["fitness_log", "precision", "generalization", "simplicity"]

def reduce_net(net, im, fm):
    net = pm4py.reduce_petri_net_invisibles(net)
    net, im, fm = pm4py.reduce_petri_net_implicit_places(net, im, fm)
    return net, im, fm

def evaluate_net(net, im, fm, eval_df):
    fit = pm4py.fitness_token_based_replay(
        eval_df, net, im, fm,
        activity_key=ACTIVITY, timestamp_key=TIMESTAMP, case_id_key=CASE_ID,
    )
    metrics = {
        "fitness_log": fit.get("log_fitness"),
        "fitness_average_trace": fit.get("average_trace_fitness"),
        "fitting_traces_percent": fit.get("percentage_of_fitting_traces"),
        "precision": pm4py.precision_token_based_replay(
            eval_df, net, im, fm,
            activity_key=ACTIVITY, timestamp_key=TIMESTAMP, case_id_key=CASE_ID,
        ),
        "generalization": pm4py.generalization_tbr(
            eval_df, net, im, fm,
            activity_key=ACTIVITY, timestamp_key=TIMESTAMP, case_id_key=CASE_ID,
        ),
        "simplicity": pm4py.simplicity_petri_net(net, im, fm),
        "f_score": None,
        "avg_score_percent": None,
    }
    f, p = metrics["fitness_log"], metrics["precision"]
    if f is not None and p is not None and (f + p) > 0:
        metrics["f_score"] = 2 * f * p / (f + p)
    available = [metrics[k] for k in DECISION_METRICS if metrics[k] is not None]
    if available:
        metrics["avg_score_percent"] = round(100 * sum(available) / len(available), 2)
    return metrics


## Directly-Follows Graph (overview)

A DFG of the full event log (`complete` events only) to get an overview of the process before
scoping and mining the individual sub-processes. **Node colour shows frequency** (darker = more
frequent), so the colour gradient highlights the main path through the flow. To keep colour free
for that purpose, the **sub-processes are distinguished structurally instead**: each flow's
activities are wrapped in a labelled, dashed box (Permit / Declaration / Request For Payment).


In [ ]:
dfg_log = pm4py.read_xes(XES_PATH)
dfg_log = pm4py.convert_to_dataframe(dfg_log)

if LIFECYCLE in dfg_log.columns:
    _complete = dfg_log[LIFECYCLE].astype(str).str.lower().eq("complete")
    if _complete.any():
        dfg_log = dfg_log[_complete].copy()

dfg_log = dfg_log[[CASE_ID, ACTIVITY, TIMESTAMP]].dropna().copy()
dfg_log[TIMESTAMP] = pd.to_datetime(dfg_log[TIMESTAMP], errors="coerce", utc=True)
dfg_log = dfg_log.dropna(subset=[TIMESTAMP])
dfg_log = dfg_log.sort_values([CASE_ID, TIMESTAMP]).reset_index(drop=True)

dfg, start_activities, end_activities = pm4py.discover_dfg(
    dfg_log, activity_key=ACTIVITY, timestamp_key=TIMESTAMP, case_id_key=CASE_ID,
)

activities_count = dfg_log[ACTIVITY].value_counts().to_dict()
dfg, start_activities, end_activities, activities_count = dfg_filtering.filter_dfg_on_activities_percentage(
    dfg, start_activities, end_activities, activities_count, 0.30,
)
dfg, start_activities, end_activities, activities_count = dfg_filtering.filter_dfg_on_paths_percentage(
    dfg, start_activities, end_activities, activities_count, 0.175,
)

dfg_gviz = dfg_visualizer.apply(
    dfg,
    variant=dfg_visualizer.Variants.FREQUENCY,
    parameters={
        DFGParam.FORMAT: "png",
        DFGParam.START_ACTIVITIES: start_activities,
        DFGParam.END_ACTIVITIES: end_activities,
        DFGParam.FONT_SIZE: 20,
    },
)

def _activity_subprocess(act):
    for sp_name, sp in SUBPROCESSES.items():
        if act.startswith(tuple(sp["prefixes"])) or act in sp.get("extra", []):
            return sp_name
    return None

node_subprocess = {}
for act in ({a for edge in dfg for a in edge} | set(start_activities) | set(end_activities)):
    sp_name = _activity_subprocess(act)
    if sp_name is not None:
        node_subprocess[str(hash(act))] = sp_name

cluster_nodes = {sp_name: [] for sp_name in SUBPROCESSES}
other_lines = []
insert_at = None
for _line in dfg_gviz.body:
    _m = re.match(r'\s*"?(-?\d+)"?\s+\[', _line)
    if _m and _m.group(1) in node_subprocess:
        if insert_at is None:
            insert_at = len(other_lines)
        cluster_nodes[node_subprocess[_m.group(1)]].append(_line.strip())
        continue
    other_lines.append(_line)

cluster_block = []
for _i, sp_name in enumerate(SUBPROCESSES):
    _lines = cluster_nodes[sp_name]
    if not _lines:
        continue
    cluster_block.append(f"subgraph cluster_{_i} {{")
    cluster_block.append(f'\tlabel="{sp_name.replace("_", " ").title()}"')
    cluster_block.append('\tstyle="rounded,dashed"')
    cluster_block.append('\tcolor="#666666"')
    cluster_block.append("\tpenwidth=2")
    cluster_block.append("\tfontsize=30")
    cluster_block.append('\tfontname="Helvetica-Bold"')
    cluster_block.append("\tlabeljust=l")
    cluster_block.extend("\t" + _l for _l in _lines)
    cluster_block.append("}")

if insert_at is None:
    insert_at = len(other_lines)
other_lines[insert_at:insert_at] = cluster_block
dfg_gviz.body = other_lines

dfg_gviz.graph_attr["rankdir"] = "TB"
for _attr in (dfg_gviz.graph_attr, dfg_gviz.node_attr, dfg_gviz.edge_attr):
    _attr["fontname"] = "Helvetica-Bold"
dfg_visualizer.save(dfg_gviz, os.path.join(OUTPUT_FOLDER, "permitlog_dfg.png"))

parsing log, completed traces :: 100%|██████████| 7065/7065 [00:01<00:00, 5373.52it/s]


''

## Preprocessing & data quality

Applied in the discovery cell before mining:

1. Keep only `complete` lifecycle events.
2. Drop exact duplicate `(case, activity, timestamp)` rows.
3. Stable event ordering: events are sorted by `(case, timestamp)` and ties on equal timestamps are broken by original log order (`ORDER`) rather than alphabetically by activity name, so no false directly-follows relations are invented.
4. Scope mining per object flow (Permit, Declaration, Request For Payment) to avoid a spaghetti model. Activities that have no flow-specific name prefix are attached to the flow they belong to: `Start trip` / `End trip` join the Permit flow (a trip is authorised by a permit), `Request Payment` / `Payment Handled` join the Request For Payment flow (they settle a request for payment), and `Send Reminder` joins the Declaration flow (it nudges the employee to submit the declaration — it is preceded by `End trip` and followed by `Declaration SUBMITTED`). This assigns every one of the log's 51 activities to exactly one sub-process.
5. Tail-step ordering: the *tail* activities (`Start trip` / `End trip`, `Request Payment` / `Payment Handled`) carry planned/booking timestamps that do not reflect their position in the control flow (e.g. a `Start trip` date can precede the permit's final approval), so they are placed after the prefix flow in their natural order and a monotonic per-case timestamp is rebuilt from that order so the miners honour it. `Send Reminder` is a *head* step (it precedes declaration submission), so it is left in its real chronological position rather than re-ordered to the tail. This stops attached activities being drawn as a disconnected parallel path.
6. Keep only complete cases (last activity contains `FINAL_APPROVED` or `REJECTED`, or the flow's trailing step `End trip` / `Payment Handled`).
7. Keep only proper-start cases (first activity contains `SUBMITTED`), removing head-truncated cases whose start falls outside the log window.
8. Rare-activity filter (`MIN_ACTIVITY_FREQ`): within each sub-process, drop activities occurring in fewer than 1% of that flow's cases (set to `0` to disable).
9. Self-loop collapsing (`COLLAPSE_SELF_LOOPS`): within each case, consecutive repeats of the same activity (`A, A → A`) are merged into a single event. Such immediate repetitions create an `A → A` edge in the directly-follows graph, which the Inductive Miner can only express as a redo loop (`A` plus a silent `τ` back-edge) and which IMf's `noise_threshold` rarely filters out. Collapsing them removes those spurious self-loops; set to `False` to keep the repetitions.



## Discovery & evaluation

- Miners: Heuristics and Inductive
- Metrics: fitness, precision, generalization, simplicity, plus F-score and their mean.
- Each sub-process is mined on samples and on all of its cases; nets are saved as PNML and the best per sub-process as PNG.


In [72]:
log = pm4py.read_xes(XES_PATH)
df = pm4py.convert_to_dataframe(log)

if LIFECYCLE in df.columns:
    complete_mask = df[LIFECYCLE].astype(str).str.lower().eq("complete")
    if complete_mask.any():
        df = df[complete_mask].copy()

df = df[[CASE_ID, ACTIVITY, TIMESTAMP]].dropna().copy()
df[TIMESTAMP] = pd.to_datetime(df[TIMESTAMP], errors="coerce", utc=True)
df = df.dropna(subset=[TIMESTAMP])
df = df.drop_duplicates(subset=[CASE_ID, ACTIVITY, TIMESTAMP]).copy()
df = df.sort_values([CASE_ID, TIMESTAMP, ACTIVITY]).reset_index(drop=True)

results = []

for sp_name, sp in SUBPROCESSES.items():
    extra_list = sp.get("extra", [])
    tail_list = sp.get("tail", [])
    prefix_mask = df[ACTIVITY].str.startswith(tuple(sp["prefixes"]))
    extra_mask = df[ACTIVITY].isin(extra_list)
    sp_df = df[prefix_mask | extra_mask].copy()

    tail_rank = {a: i for i, a in enumerate(tail_list)}
    sp_df["_phase"] = sp_df[ACTIVITY].isin(tail_list).astype(int)
    sp_df["_rank"] = sp_df[ACTIVITY].map(tail_rank).fillna(-1).astype(int)
    order_cols = [CASE_ID, "_phase", "_rank", TIMESTAMP, ACTIVITY]
    sp_df = sp_df.sort_values(order_cols).reset_index(drop=True)

    if sp.get("start"):
        first_act = sp_df.groupby(CASE_ID)[ACTIVITY].first()
        start_pattern = "|".join(re.escape(s) for s in sp["start"])
        keep_cases = first_act[first_act.str.contains(start_pattern, regex=True)].index
        sp_df = sp_df[sp_df[CASE_ID].isin(keep_cases)].copy()

    if sp.get("end"):
        last_act = sp_df.groupby(CASE_ID)[ACTIVITY].last()
        end_pattern = "|".join(re.escape(e) for e in sp["end"])
        keep_cases = last_act[last_act.str.contains(end_pattern, regex=True)].index
        sp_df = sp_df[sp_df[CASE_ID].isin(keep_cases)].copy()

    if sp_df.empty:
        continue

    if MIN_ACTIVITY_FREQ > 0:
        cases_per_activity = sp_df.groupby(ACTIVITY)[CASE_ID].nunique()
        min_cases = MIN_ACTIVITY_FREQ * sp_df[CASE_ID].nunique()
        keep_activities = cases_per_activity[cases_per_activity >= min_cases].index
        sp_df = sp_df[sp_df[ACTIVITY].isin(keep_activities)].copy()

    if sp_df.empty:
        continue

    sp_df = sp_df.sort_values(order_cols).reset_index(drop=True)

    if COLLAPSE_SELF_LOOPS:
        same_as_prev = (
            sp_df[CASE_ID].eq(sp_df[CASE_ID].shift())
            & sp_df[ACTIVITY].eq(sp_df[ACTIVITY].shift())
        )
        sp_df = sp_df[~same_as_prev].reset_index(drop=True)

    seq = sp_df.groupby(CASE_ID).cumcount()
    sp_df[TIMESTAMP] = pd.Timestamp("2000-01-01", tz="UTC") + pd.to_timedelta(seq, unit="m")
    sp_df = sp_df.drop(columns=["_phase", "_rank"])

    all_cases = (
        pd.Series(sp_df[CASE_ID].unique()).sample(frac=1, random_state=42).tolist()
    )
    total_cases = len(all_cases)

    case_sample_sizes = [x for x in BASE_SAMPLE_SIZES if x < total_cases]
    case_sample_sizes.append(total_cases)

    for sample_size in case_sample_sizes:
        selected_cases = all_cases[:sample_size]
        eval_df = sp_df[sp_df[CASE_ID].isin(selected_cases)].copy()
        eval_df = eval_df.sort_values([CASE_ID, TIMESTAMP]).reset_index(drop=True)

        for setting in heuristics_settings:
            label = (
                sp_name + "_heuristics_cases_" + str(sample_size)
                + "_dep_" + str(setting["dependency_threshold"]).replace(".", "_")
                + "_and_" + str(setting["and_threshold"]).replace(".", "_")
                + "_loop2_" + str(setting["loop_two_threshold"]).replace(".", "_")
            )
            net, im, fm = pm4py.discover_petri_net_heuristics(eval_df, **setting)
            net, im, fm = reduce_net(net, im, fm)
            metrics = evaluate_net(net, im, fm, eval_df)
            pm4py.write_pnml(net, im, fm, os.path.join(OUTPUT_FOLDER, label + ".pnml"))
            results.append({
                "subprocess": sp_name,
                "algorithm": "Heuristics Miner",
                "sample_cases": sample_size,
                "sample_events": len(eval_df),
                "dependency_threshold": setting["dependency_threshold"],
                "and_threshold": setting["and_threshold"],
                "loop_two_threshold": setting["loop_two_threshold"],
                "noise_threshold": None,
                **metrics,
            })

        for setting in inductive_settings:
            label = (
                sp_name + "_inductive_cases_" + str(sample_size)
                + "_noise_" + str(setting["noise_threshold"]).replace(".", "_")
            )
            net, im, fm = pm4py.discover_petri_net_inductive(eval_df, **setting)
            net, im, fm = reduce_net(net, im, fm)
            metrics = evaluate_net(net, im, fm, eval_df)
            pm4py.write_pnml(net, im, fm, os.path.join(OUTPUT_FOLDER, label + ".pnml"))
            results.append({
                "subprocess": sp_name,
                "algorithm": "Inductive Miner",
                "sample_cases": sample_size,
                "sample_events": len(eval_df),
                "dependency_threshold": None,
                "and_threshold": None,
                "loop_two_threshold": None,
                "noise_threshold": setting["noise_threshold"],
                **metrics,
            })

results_df = pd.DataFrame(results)
results_path = os.path.join(OUTPUT_FOLDER, "permitlog_mining_quality_results.csv")


replaying log with TBR, completed traces :: 100%|██████████| 141/141 [00:00<00:00, 1670.18it/s]


## Performance overlay (green → red throughput)

For each best model we overlay the **average throughput time** on every activity block and colour it
from **green (fastest)** to **red (slowest)** with the `RdYlGn_r` colormap.

- The throughput of a block is the mean waiting time leading **into** that activity (time since the
  previous step of the same flow), averaged over all cases.
- pm4py has no built-in green→red Petri-net heatmap, so we feed a custom `decorations` dict
  (per-transition `label` + `color`) straight to the renderer.
- Times are measured on the **original** timestamps — the discovery step rewrites timestamps to a
  synthetic 1-minute spacing to fix ordering, which would otherwise erase real durations.
- *Caveat:* `Start trip` / `End trip` / `Payment Handled` carry planned dates that can precede the
  prior step, so negative gaps are clipped to 0 and their times are noisier than the permit steps.


In [ ]:
scored = results_df.dropna(subset=["avg_score_percent"])
best_per_algo = scored.loc[
    scored.groupby(["subprocess", "algorithm"])["avg_score_percent"].idxmax()
].sort_values(["subprocess", "algorithm"])

raw = log if isinstance(log, pd.DataFrame) else pm4py.convert_to_dataframe(log)
raw = raw.copy()
if LIFECYCLE in raw.columns:
    _complete = raw[LIFECYCLE].astype(str).str.lower().eq("complete")
    if _complete.any():
        raw = raw[_complete]
raw = raw[[CASE_ID, ACTIVITY, TIMESTAMP]].dropna().copy()
raw[TIMESTAMP] = pd.to_datetime(raw[TIMESTAMP], errors="coerce", utc=True)
raw = raw.dropna(subset=[TIMESTAMP]).sort_values([CASE_ID, TIMESTAMP]).reset_index(drop=True)
CMAP = matplotlib.colormaps["RdYlGn_r"]

def human_time(seconds):
    s = float(seconds)
    for unit, size in (("d", 86400.0), ("h", 3600.0), ("m", 60.0)):
        if s >= size:
            return f"{s / size:.1f}{unit}"
    return f"{s:.0f}s"

def activity_throughput(prefixes, extra):
    mask = raw[ACTIVITY].str.startswith(tuple(prefixes)) | raw[ACTIVITY].isin(extra)
    sub = raw[mask].copy()
    sub["_delta"] = sub.groupby(CASE_ID)[TIMESTAMP].diff().dt.total_seconds()
    sub = sub.dropna(subset=["_delta"])
    sub["_delta"] = sub["_delta"].clip(lower=0.0)
    return sub.groupby(ACTIVITY)["_delta"].mean()

def best_model_label(row):
    scope = row["subprocess"]
    n = int(row["sample_cases"])
    if row["algorithm"] == "Heuristics Miner":
        return (f"{scope}_heuristics_cases_{n}"
                f"_dep_{str(float(row['dependency_threshold'])).replace('.', '_')}"
                f"_and_{str(float(row['and_threshold'])).replace('.', '_')}"
                f"_loop2_{str(float(row['loop_two_threshold'])).replace('.', '_')}")
    return (f"{scope}_inductive_cases_{n}"
            f"_noise_{str(float(row['noise_threshold'])).replace('.', '_')}")

for _, row in best_per_algo.iterrows():
    sp = SUBPROCESSES[row["subprocess"]]
    thr = activity_throughput(sp["prefixes"], sp.get("extra", []))
    if thr.empty:
        continue
    vmin, vmax = float(thr.min()), float(thr.max())
    norm = matplotlib.colors.Normalize(vmin=vmin, vmax=vmax if vmax > vmin else vmin + 1.0)

    label = best_model_label(row)
    net, im, fm = pm4py.read_pnml(os.path.join(OUTPUT_FOLDER, label + ".pnml"))

    decorations = {}
    for t in net.transitions:
        if t.label is None:
            continue
        if t.label in thr.index:
            val = float(thr[t.label])
            decorations[t] = {
                "label": f"{t.label}\n{human_time(val)}",
                "color": matplotlib.colors.to_hex(CMAP(norm(val))),
            }
        else:
            decorations[t] = {
                "label": f"{t.label}\n(start)",
                "color": matplotlib.colors.to_hex(CMAP(0.0)),
            }

    gviz = pn_visualizer.apply(
        net, im, fm,
        variant=pn_visualizer.Variants.WO_DECORATION,
        parameters={
            PNParam.FORMAT: "png",
            PNParam.DECORATIONS: decorations,
            PNParam.RANKDIR: "TB",
            PNParam.FONT_SIZE: 20,
        },
    )
    for _attr in (gviz.graph_attr, gviz.node_attr, gviz.edge_attr):
        _attr["fontname"] = "Helvetica-Bold"
    pn_visualizer.save(gviz, os.path.join(OUTPUT_FOLDER, label + ".png"))

best_per_algo


,subprocess,algorithm,sample_cases,sample_events,dependency_threshold,and_threshold,loop_two_threshold,noise_threshold,fitness_log,fitness_average_trace,fitting_traces_percent,precision,generalization,simplicity,f_score,avg_score_percent
42,declaration,Heuristics Miner,5179,22261,0.2,0.4,0.2,NaN,0.956290,0.954146,54.875459,0.939177,0.747749,0.577982,0.947656,80.53
29,declaration,Inductive Miner,250,1061,NaN,NaN,NaN,0.1,0.992095,0.988888,88.800000,0.607645,0.826614,0.704918,0.753675,78.28
19,permit,Heuristics Miner,7050,38128,0.3,0.5,0.3,NaN,0.959139,0.958387,53.574468,0.993627,0.815782,0.623529,0.976078,84.80
9,permit,Inductive Miner,500,2725,NaN,NaN,NaN,0.0,1.000000,1.000000,100.000000,0.752461,0.866498,0.660377,0.858748,81.98
66,request_for_payment,Heuristics Miner,1313,8941,0.2,0.4,0.2,NaN,0.973805,0.974900,77.760853,0.942669,0.679045,0.586957,0.957984,79.56
53,request_for_payment,Inductive Miner,250,1708,NaN,NaN,NaN,0.1,0.999279,0.998654,98.800000,0.566294,0.827219,0.696970,0.722912,77.24
